# Multiplets of visits (singlets, doublets, triplets, ...) versus survey mode, in baseline survey v5.3

In the OpSim simulations the visits are not isolated: the scheduler observes the same pointing several times during a night, in the same or in different filters (a **pair** `iz` or `gr` 33 minutes apart, a **triplet**, a long DDF sequence, a single `greedy` visit, ...). This notebook builds these **multiplets** directly from the visits and asks: **is a given type of multiplet associated with a given survey mode (WFD, DDF, mini-surveys, micro-surveys)?** It also shows **where on the sky** each type of multiplet is observed.

- creation date: 2026-10-14
- companion notebooks: `01_ExploreSims/01_ExampleReadOpsimDb` (figure of `scheduler_note` by band), `01_ExploreSims/03_OpsimStat`, `07_WFDMiniMicroS_NVisitsVsTime`, `10_FilterBalance` (same definition of the survey modes)
- survey modes definition: https://github.com/lsst/survey-strategy_lsst_io

## Goal, definitions and outputs

**Multiplet.** All the visits of **one night** whose pointing centres are closer than `MATCH_RADIUS_DEG` form one multiplet. For the DDFs the dithering between visits of a night is removed for the matching (all the visits of one DDF in one night form one multiplet). The definition is purely data-driven: it does not rely on how the scheduler named the sequences. The `scheduler_note` is only used afterwards, as a cross-check (section 5).

**Multiplicity class** (number of visits of the multiplet): `1` singlet, `2` doublet (pair), `3` triplet, `4` quadruplet, `5+` five visits or more (typically DDF sequences).

**Band combination**: the bands of the visits of the multiplet, sorted in the order u, g, r, i, z, y (repeats are kept: `rr` is two visits in r). The time-ordered string (`bands_seq`) is kept in the table.

**Survey mode** (each multiplet is assigned to the mode of its visits, same definition as `10_FilterBalance`):

| Mode | Definition |
|---|---|
| `WFD` | pixels of the scheduler map: low dust (+ euclid overlap), bulge, Virgo, SMC-LMC |
| `DDF` | visits with a `DD` scheduler note; the finer `region` is the DDF name |
| `Mini-surveys` | `NES`, `SCP`, `MW non-WFD` (and any other label of the map) |
| `Micro-surveys (no DDF)` | `RGES`, `TwilightNEO`, `ToO` |

**Questions and representations**

1. *What are the multiplets?* Tables and bar plots of the number of multiplets per class and per band combination (Figures 1, 2, 4, 5).
2. *Is the type of multiplet associated with the survey mode?* Fractions in both directions (type given the mode, mode given the type), chi-square test of independence, Cramer V (effect size) and standardised residuals (Figure 3).
3. *How are the multiplets spread in time?* Time between successive visits of a multiplet (Figure 6) and number of multiplets per year (Figure 7).
4. *Do the multiplets follow the scheduler sequences?* Cross-check with the `scheduler_note` family (Figure 8).
5. *Where are they?* HEALPix maps of the number of multiplets covering each pixel, per class (Figure 9), and of the fraction of each class and the mean number of visits per multiplet (Figure 10).

**Input**: the OpSim database `baseline_v5.3.6_10yrs.db` (variable `opsdb`).

**Outputs**: the tables (`.csv`) go to `DATA13_Multiplets/`; all figures (`.png` and `.pdf`) go to `FIG13_Multiplets/`.

## 0. Setup

In [ ]:
import os
import sqlite3
import itertools
import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

from rubin_scheduler.utils import SURVEY_START_MJD
from rubin_scheduler.scheduler.utils import CurrentAreaMap

In [ ]:
plt.rcParams.update(
    {
        "font.size": 14,
        "axes.labelsize": 16,
        "axes.titlesize": 16,
        "xtick.labelsize": 14,
        "ytick.labelsize": 14,
        "legend.fontsize": 14,
        "legend.title_fontsize": 14,
    }
)

In [ ]:
# Output directories: tables go to DATA_DIR, figures go to FIG_DIR
DATA_DIR = "DATA13_Multiplets"
FIG_DIR = "FIG13_Multiplets"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# OpSim database. Replace by another path to analyse another run.
opsdb = "/Users/dagoret/DATA/OpSim/baseline_v5.3.6_10yrs.db"
opsim_name = os.path.split(opsdb)[-1].replace(".db", "")
print(opsdb)

YEAR = 365.25  # days
N_YEARS = 10
BANDS = list("ugrizy")
BAND_COLORS = {
    "u": "#0c71ff",
    "g": "#49be61",
    "r": "#c61c00",
    "i": "#ffc200",
    "z": "#f341a2",
    "y": "#5d0000",
}
NSIDE = 64  # HEALPix resolution of the scheduler map and of the multiplet maps
NPIX = hp.nside2npix(NSIDE)

# Multiplet definition and sky coverage
MATCH_RADIUS_DEG = 0.5  # visits of the same night closer than this belong to the same multiplet
FOV_RADIUS_DEG = 1.75  # radius of the field of view, used to find the pixels covered by a visit

# Display thresholds
MIN_MULTIPLETS_GROUP = 500  # modes / regions with fewer multiplets are removed from the plots
TOP_COMBOS = 10  # number of band combinations shown per mode in the bar plots
MIN_TOTAL_PIX = 20  # minimum number of multiplets in a pixel to compute a fraction in the maps
MAP_MODE = None  # None = maps of all the multiplets; or one of the modes below, e.g. 'WFD'

# Region definition: name -> list of labels of the scheduler map (as in 10_FilterBalance).
# 'euclid_overlap' is merged with 'lowdust'.
WFD_REGIONS = {
    "low dust": ["lowdust", "euclid_overlap"],
    "bulge": ["bulgy"],
    "virgo": ["virgo"],
    "SMC-LMC": ["LMC_SMC"],
}
MINI_SURVEYS = {
    "NES": ["nes"],
    "SCP": ["scp"],
    "MW non-WFD": ["dusty_plane"],
}

MODES = ["WFD", "DDF", "Mini-surveys", "Micro-surveys (no DDF)", "Unassigned"]
MODE_COLORS = {
    "WFD": "#1f77b4",
    "DDF": "#d62728",
    "Mini-surveys": "#2ca02c",
    "Micro-surveys (no DDF)": "#ff7f0e",
    "Unassigned": "#7f7f7f",
}

# Multiplicity classes (number of visits of the multiplet)
CLASS_ORDER = ["1", "2", "3", "4", "5+"]
CLASS_LABELS = {
    "1": "singlet (1)",
    "2": "doublet (2)",
    "3": "triplet (3)",
    "4": "quadruplet (4)",
    "5+": "5 visits or more",
}
CLASS_COLORS = {
    "1": "#4c72b0",
    "2": "#dd8452",
    "3": "#55a868",
    "4": "#c44e52",
    "5+": "#8172b3",
}

## 1. Survey modes and loading of the visits

`CurrentAreaMap` returns one survey-mode label per HEALPix pixel. A label present in the map but not listed in `WFD_REGIONS` or `MINI_SURVEYS` is automatically added as a mini-survey.

All visits are read once and sorted in time. Time is expressed in years since the survey start (`SURVEY_START_MJD`).

- **Micro-surveys and DDFs**: selected from `scheduler_note` (`ToO`, `twilight_near_sun`, `RGES`, `DD`), with the priority ToO > TwilightNEO > RGES > DDF.
- **Other visits**: mode and region from the label of the HEALPix pixel containing the visit centre.
- **Scheduler family**: first word of the `scheduler_note` (`pair_33`, `blob_long`, `greedy`, `templates`, `twilight_near_sun`, `DDF`, `ToO`, `RGES`), used in section 5.

In [ ]:
# Scheduler footprint: one survey-mode label per HEALPix pixel
skymap = CurrentAreaMap(nside=NSIDE)
footprint_maps, labels = skymap.return_maps()
labels = np.asarray(labels).astype(str)
label_names = [l for l in np.unique(labels) if l != ""]
print("Labels in the target map:", label_names)

# Labels configured above but absent from the map
for name, labs in {**WFD_REGIONS, **MINI_SURVEYS}.items():
    for l in labs:
        if l not in label_names:
            print(f"WARNING: label {l} of region {name} is not in the target map")

# Labels of the map not configured above: added as mini-surveys
used_labels = {l for labs in list(WFD_REGIONS.values()) + list(MINI_SURVEYS.values()) for l in labs}
for l in label_names:
    if l not in used_labels:
        print(f"Label {l} is not in WFD_REGIONS or MINI_SURVEYS: added as a mini-survey")
        MINI_SURVEYS[l] = [l]

In [ ]:
query = (
    "select observationStartMJD, night, band, fieldRA, fieldDec, "
    "visitExposureTime, scheduler_note from observations"
)
conn = sqlite3.connect(opsdb)
visits = pd.read_sql(query, conn)
conn.close()

visits = visits.sort_values("observationStartMJD").reset_index(drop=True)
visits["years"] = (visits["observationStartMJD"] - SURVEY_START_MJD) / YEAR
N_ALL = len(visits)
duration = visits["years"].max()
print(f"{N_ALL} visits in the survey, duration {duration:.3f} years")

# Micro-surveys and DDFs from scheduler_note (one tag per visit)
note = visits["scheduler_note"].fillna("")
is_too = note.str.contains("ToO").to_numpy()
is_twilight = note.str.contains("twilight_near_sun").to_numpy()
is_rges = note.str.contains("RGES").to_numpy()
is_ddf = note.str.contains("DD").to_numpy() & ~is_rges
visits["micro"] = np.select(
    [is_too, is_twilight, is_rges, is_ddf],
    ["ToO", "TwilightNEO", "RGES", "DDF"],
    default="",
)
micro = visits["micro"].to_numpy(dtype=object)
is_micro = micro != ""

# Scheduler family: first word of the scheduler_note
family = note.str.extract("^(pair_[0-9]+|blob_long|blob|greedy|templates|twilight_near_sun)")[0]
family = family.fillna("other").to_numpy(dtype=object)
family[micro == "RGES"] = "RGES"
family[micro == "DDF"] = "DDF"
family[micro == "ToO"] = "ToO"
visits["family"] = family
print("Scheduler families:")
print(visits["family"].value_counts().to_string())
print("Notes not recognised (family other), top 10:")
print(visits.loc[visits["family"] == "other", "scheduler_note"].value_counts().head(10).to_string())

# Label of the HEALPix pixel containing the visit centre (RA, Dec in degrees)
pix = hp.ang2pix(NSIDE, visits["fieldRA"].to_numpy(), visits["fieldDec"].to_numpy(), lonlat=True)
visits["label"] = labels[pix]
print(visits["band"].value_counts().to_string())

**Mode and region of each visit.** `mode` is one of the four survey modes (or `Unassigned` for a visit outside every region of the map); `region` is the finer label: low dust, bulge, virgo, SMC-LMC, NES, SCP, MW non-WFD, `RGES`, `TwilightNEO`, `ToO`, or `DDF:<name>` for each DDF.

In [ ]:
sky_label = visits["label"].to_numpy(dtype=object)
not_micro = ~is_micro
mode = np.full(N_ALL, "Unassigned", dtype=object)
region = np.full(N_ALL, "Unassigned", dtype=object)

for name, labs in WFD_REGIONS.items():
    m = not_micro & np.isin(sky_label, labs)
    mode[m] = "WFD"
    region[m] = name
for name, labs in MINI_SURVEYS.items():
    m = not_micro & np.isin(sky_label, labs)
    mode[m] = "Mini-surveys"
    region[m] = name
for tag in ["RGES", "TwilightNEO", "ToO"]:
    m = micro == tag
    mode[m] = "Micro-surveys (no DDF)"
    region[m] = tag

# DDFs: one region per DDF, named from the scheduler_note (DD:<name>)
ddf_name = note.str.extract("DD:[ ]*([A-Za-z0-9_]+)")[0].fillna("unknown").to_numpy(dtype=object)
m_ddf = micro == "DDF"
mode[m_ddf] = "DDF"
region[m_ddf] = np.array(["DDF:" + s for s in ddf_name[m_ddf]], dtype=object)

visits["mode"] = mode
visits["region"] = region
print(visits["mode"].value_counts().to_string())
print(visits.groupby(["mode", "region"]).size().to_string())

## 2. Building the multiplets

Each visit is a point `(x, y, z, 10 * night)`: `(x, y, z)` is the unit vector of the pointing centre and the fourth coordinate separates the nights (two visits of different nights are always more than 10 apart, hence never linked). All the pairs of visits closer than `MATCH_RADIUS_DEG` (chord distance on the sphere) are linked with a KD-tree, and the **connected components** of this graph are the multiplets. For the DDFs, the unit vector of every visit is first replaced by the median position of its DDF, so that the dithering of the DDF visits within a night does not split a sequence.

The radius is much smaller than the distance between two neighbouring fields (a few degrees), so two different fields are never merged. If the survey includes dithers larger than the radius within a night for the non-DDF fields, a multiplet would be split: check the distribution of the number of visits per multiplet printed below.

In [ ]:
ra_rad = np.radians(visits["fieldRA"].to_numpy())
dec_rad = np.radians(visits["fieldDec"].to_numpy())
xyz = np.column_stack([np.cos(dec_rad) * np.cos(ra_rad), np.cos(dec_rad) * np.sin(ra_rad), np.sin(dec_rad)])

# DDF: remove the dithering for the matching (median position of each DDF)
for name in np.unique(region[m_ddf]):
    sel = region == name
    c = np.median(xyz[sel], axis=0)
    xyz[sel] = c / np.linalg.norm(c)

night = visits["night"].to_numpy(dtype=float)
points = np.column_stack([xyz, 10.0 * night])
tree = cKDTree(points)
chord = 2.0 * np.sin(np.radians(MATCH_RADIUS_DEG) / 2.0)
pairs = tree.query_pairs(r=chord, output_type="ndarray")
print(f"{len(pairs)} pairs of visits closer than {MATCH_RADIUS_DEG} deg in the same night")

graph = coo_matrix((np.ones(len(pairs), dtype=np.int8), (pairs[:, 0], pairs[:, 1])), shape=(N_ALL, N_ALL))
n_mult, cid = connected_components(graph, directed=False)
visits["cid"] = cid
print(f"{n_mult} multiplets for {N_ALL} visits")

**One row per multiplet.** The table `mult` gives, for each multiplet: the number of visits `n` and its class `cls`, the time-ordered bands `bands_seq`, the bands sorted in the order u, g, r, i, z, y `bands_sorted`, the pointing of the first visit (`ra`, `dec`, in degrees), the night, the time `span_min` between first and last visit (minutes), the mode and region, and the scheduler families of its visits (`families`, e.g. `pair_33` or `blob_long+pair_33` for a triplet made of a pair and a later visit). `dt_prev_min` (time since the previous visit of the same multiplet) is added to the visit table.

In [ ]:
vis = visits.sort_values(["cid", "observationStartMJD"]).reset_index(drop=True)
vis["dt_prev_min"] = vis.groupby("cid")["observationStartMJD"].diff() * 1440.0
g = vis.groupby("cid", sort=False)

mult = pd.DataFrame(
    {
        "n": g.size(),
        "t0": g["observationStartMJD"].min(),
        "t1": g["observationStartMJD"].max(),
        "years": g["years"].min(),
        "night": g["night"].first(),
        "ra": g["fieldRA"].first(),
        "dec": g["fieldDec"].first(),
        "mode": g["mode"].first(),
        "region": g["region"].first(),
        "n_modes": g["mode"].nunique(),
        "bands_seq": g["band"].agg("".join),
        "families": g["family"].agg(lambda s: "+".join(sorted(set(s)))),
    }
)
mult["span_min"] = (mult["t1"] - mult["t0"]) * 1440.0


def band_key(b):
    return BANDS.index(b) if b in BANDS else 99


sorted_map = {s: "".join(sorted(s, key=band_key)) for s in mult["bands_seq"].unique()}
mult["bands_sorted"] = mult["bands_seq"].map(sorted_map)
mult["cls"] = np.where(mult["n"] >= 5, "5+", mult["n"].astype(str))

print(f'{len(mult)} multiplets, {mult["n"].sum()} visits (survey: {N_ALL})')
assert mult["n"].sum() == N_ALL
print(
    f'Multiplets with visits in two different modes (assigned to the first visit): {(mult["n_modes"] > 1).sum()}'
)
print("Number of visits per multiplet:")
print(mult["n"].value_counts().sort_index().head(15).to_string())

## 3. Number of multiplets per class and per survey mode

`ct_mult` counts the **multiplets**, `ct_vis` counts the **visits** they contain (a DDF multiplet of 8 visits counts 8). The same tables are built per `region` (`ct_reg`, `ct_reg_vis`). The summary table gives, for each mode: the number of multiplets in each class, the **percentage of the mode** (type given the mode: how is the mode made?) and the **percentage of the class** (mode given the type: where do the doublets, triplets, ... come from?). All tables are saved in `DATA13_Multiplets/`.

In [ ]:
MODES_USED = [m for m in MODES if (mult["mode"] == m).sum() >= MIN_MULTIPLETS_GROUP]
print("Modes kept:", MODES_USED, "| removed (too few multiplets):", [m for m in MODES if m not in MODES_USED])

reg_counts = mult["region"].value_counts()
region_mode = mult.groupby("region")["mode"].first().to_dict()
REGIONS_USED = [r for r in reg_counts.index if reg_counts[r] >= MIN_MULTIPLETS_GROUP]
REGIONS_USED = sorted(REGIONS_USED, key=lambda r: (MODES.index(region_mode[r]), -reg_counts[r]))


def count_tables(group_col, groups):
    ct_m = pd.crosstab(mult[group_col], mult["cls"]).reindex(index=groups, columns=CLASS_ORDER, fill_value=0)
    ct_v = (
        mult.groupby([group_col, "cls"])["n"]
        .sum()
        .unstack(fill_value=0)
        .reindex(index=groups, columns=CLASS_ORDER, fill_value=0)
    )
    return ct_m, ct_v


ct_mult, ct_vis = count_tables("mode", MODES_USED)
ct_reg, ct_reg_vis = count_tables("region", REGIONS_USED)
ct_mult.to_csv(os.path.join(DATA_DIR, f"{opsim_name}_multiplets_per_mode.csv"))
ct_vis.to_csv(os.path.join(DATA_DIR, f"{opsim_name}_visits_per_mode_and_class.csv"))
ct_reg.to_csv(os.path.join(DATA_DIR, f"{opsim_name}_multiplets_per_region.csv"))
ct_reg_vis.to_csv(os.path.join(DATA_DIR, f"{opsim_name}_visits_per_region_and_class.csv"))

summary = pd.concat(
    {
        "N multiplets": ct_mult,
        "% of the mode": ct_mult.div(ct_mult.sum(axis=1), axis=0).mul(100).round(1),
        "% of the class": ct_mult.div(ct_mult.sum(axis=0), axis=1).mul(100).round(1),
    },
    axis=1,
)
summary.to_csv(os.path.join(DATA_DIR, f"{opsim_name}_summary_per_mode.csv"))
summary

In [ ]:
def save_fig(fig, stem):
    # Each figure is saved as .png and .pdf in FIG_DIR
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{opsim_name}_{stem}.{ext}"), dpi=150, bbox_inches="tight")


def grid_axes(n, ncols=3, sharex=False):
    # Grid of n panels (3 columns); the extra panels are hidden
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.5 * ncols, 4.5 * nrows), sharex=sharex, squeeze=False)
    axes = axes.ravel()
    for ax in axes[n:]:
        ax.axis("off")
    return fig, axes


def stacked_class_bars(ax, ct, title, xlabel):
    # ct: counts, index = groups (modes or regions), columns = CLASS_ORDER. Each row is normalised to 100%.
    frac = 100 * ct.div(ct.sum(axis=1), axis=0)
    y = np.arange(len(ct))
    left = np.zeros(len(ct))
    for c in CLASS_ORDER:
        w = frac[c].to_numpy()
        ax.barh(y, w, left=left, color=CLASS_COLORS[c], edgecolor="k", linewidth=0.4, label=CLASS_LABELS[c])
        for yi in range(len(ct)):
            if w[yi] >= 5:
                ax.text(
                    left[yi] + w[yi] / 2,
                    yi,
                    f"{w[yi]:.0f}%",
                    ha="center",
                    va="center",
                    color="w",
                    fontsize=11,
                    fontweight="bold",
                )
        left = left + w
    ax.set_yticks(y)
    ax.set_yticklabels([f"{grp} ({ct.loc[grp].sum():,})" for grp in ct.index])
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    ax.grid(alpha=0.3, axis="x")

### Figure 1 - Multiplet classes in each survey mode (`<run>_classes_per_mode.png`)

One bar per survey mode, the colours are the multiplicity classes. **Left**: fraction of the *multiplets* of the mode in each class. **Right**: fraction of the *visits* of the mode contained in each class (a long DDF sequence weighs more in visits than in multiplets). The total number of multiplets (left) or visits (right) is written next to the mode name.

In [ ]:
fig1, (axa, axb) = plt.subplots(1, 2, figsize=(20, 0.9 * len(MODES_USED) + 3.5))
stacked_class_bars(axa, ct_mult, "Multiplets", "Fraction of the multiplets of the mode (%)")
stacked_class_bars(axb, ct_vis, "Visits", "Fraction of the visits of the mode (%)")
handles, names = axa.get_legend_handles_labels()
fig1.legend(
    handles,
    names,
    ncol=len(CLASS_ORDER),
    loc="lower center",
    bbox_to_anchor=(0.5, -0.03),
    title="visits per multiplet",
)
fig1.suptitle(f"Multiplet classes per survey mode - {opsim_name}")
fig1.tight_layout(rect=(0, 0.1, 1, 0.94))
save_fig(fig1, "classes_per_mode")
plt.show()

### Figure 2 - Same, per region (`<run>_classes_per_region.png`)

Same as Figure 1 with the finer `region` label: the four regions of the WFD, the mini-surveys, the micro-surveys and each DDF separately. Regions with fewer than `MIN_MULTIPLETS_GROUP` multiplets are not shown.

In [ ]:
fig2, (axa, axb) = plt.subplots(1, 2, figsize=(20, 0.55 * len(REGIONS_USED) + 3.5))
stacked_class_bars(axa, ct_reg, "Multiplets", "Fraction of the multiplets of the region (%)")
stacked_class_bars(axb, ct_reg_vis, "Visits", "Fraction of the visits of the region (%)")
handles, names = axa.get_legend_handles_labels()
fig2.legend(
    handles,
    names,
    ncol=len(CLASS_ORDER),
    loc="lower center",
    bbox_to_anchor=(0.5, -0.03),
    title="visits per multiplet",
)
fig2.suptitle(f"Multiplet classes per region - {opsim_name}")
fig2.tight_layout(rect=(0, 0.06, 1, 0.95))
save_fig(fig2, "classes_per_region")
plt.show()

## 4. Is the type of multiplet associated with the survey mode?

### Figure 3 - Standardised residuals of the (mode x class) table (`<run>_association_mode_class.png`)

If the multiplicity class were **independent** of the survey mode, the number of multiplets of the mode `m` in the class `c` would be `E = N_m N_c / N`. For each cell the figure gives the **standardised residual** `(O - E) / sqrt(E)` (large number, bold) and the observed number `O` (small number). **Red** = more multiplets of this class in this mode than expected, **blue** = fewer.

The title gives **Cramer V**, an effect size between 0 (independence) and 1 (each class lives in one mode only). With hundreds of thousands of multiplets the chi-square p-value is always tiny, so **V and the residuals matter, not the p-value** (saved in `..._association_tests.csv`). Left: per mode; right: per region.

In [ ]:
def assoc_heatmap(ax, ct, title):
    obs = ct.to_numpy(dtype=float)
    keep_r = obs.sum(axis=1) > 0
    keep_c = obs.sum(axis=0) > 0
    o = obs[keep_r][:, keep_c]
    chi2, p, dof, expd = stats.chi2_contingency(o)
    v = np.sqrt(chi2 / (o.sum() * (min(o.shape) - 1)))
    resid = (o - expd) / np.sqrt(expd)
    vmax = np.max(np.abs(resid))
    im = ax.imshow(resid, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    for i in range(o.shape[0]):
        for j in range(o.shape[1]):
            ax.text(
                j, i - 0.12, f"{resid[i, j]:+.0f}", ha="center", va="center", fontsize=12, fontweight="bold"
            )
            ax.text(j, i + 0.2, f"{o[i, j]:,.0f}", ha="center", va="center", fontsize=8)
    ax.set_xticks(range(o.shape[1]))
    ax.set_xticklabels([CLASS_LABELS[c] for c in ct.columns[keep_c]], rotation=20)
    ax.set_yticks(range(o.shape[0]))
    ax.set_yticklabels(list(ct.index[keep_r]))
    ax.set_title(f"{title} (Cramer V = {v:.2f})")
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02, label="standardised residual (sigma)")
    return {"chi2": chi2, "dof": dof, "p-value": p, "Cramer V": v}


fig3, (axa, axb) = plt.subplots(1, 2, figsize=(20, 0.45 * len(REGIONS_USED) + 4))
res_mode = assoc_heatmap(axa, ct_mult, "Survey mode x class")
res_reg = assoc_heatmap(axb, ct_reg, "Region x class")
fig3.suptitle(f"Association between survey mode and multiplet class - {opsim_name}")
fig3.tight_layout()
save_fig(fig3, "association_mode_class")
plt.show()

tests = pd.DataFrame({"mode x class": res_mode, "region x class": res_reg}).T
tests.to_csv(os.path.join(DATA_DIR, f"{opsim_name}_association_tests.csv"))
tests

## 5. Which band combinations?

### Figure 4 - Most frequent band combinations per survey mode (`<run>_band_combos_<class>.png`)

One figure per multiplicity class (`1`, `2`, `3`, `4`, `5+`), one panel per survey mode. Each bar is a band combination (bands sorted u, g, r, i, z, y; `rr` = two visits in r), with the number of multiplets and the percentage of the multiplets of this class **in this mode**. For the singlets the bar colour is the filter colour, otherwise it is the mode colour. The full tables (all combinations) are saved as `..._band_combos_<class>.csv`.

In [ ]:
def combo_table(cls_value, group_col, groups):
    sub = mult[mult["cls"] == cls_value]
    return pd.crosstab(sub["bands_sorted"], sub[group_col]).reindex(columns=groups, fill_value=0)


for c in CLASS_ORDER:
    tab = combo_table(c, "mode", MODES_USED)
    tab.to_csv(os.path.join(DATA_DIR, f'{opsim_name}_band_combos_{c.replace("+", "plus")}.csv'))
    fig, axes = grid_axes(len(MODES_USED))
    for ax, m in zip(axes, MODES_USED):
        col = tab[m].sort_values(ascending=False)
        col = col[col > 0].head(TOP_COMBOS)
        if len(col) == 0:
            ax.axis("off")
            continue
        tot = tab[m].sum()
        colors = [BAND_COLORS.get(k, "grey") for k in col.index] if c == "1" else MODE_COLORS[m]
        ypos = np.arange(len(col))
        ax.barh(ypos, col.to_numpy(), color=colors, edgecolor="k", linewidth=0.5)
        for yi, val in zip(ypos, col.to_numpy()):
            ax.text(val, yi, f" {val:,} ({100 * val / tot:.0f}%)", va="center", fontsize=10)
        ax.set_xlim(0, 1.45 * col.max())
        ax.set_yticks(ypos)
        ax.set_yticklabels(list(col.index))
        ax.invert_yaxis()
        ax.set_title(f"{m} ({tot:,})")
        ax.set_xlabel("Number of multiplets")
        ax.grid(alpha=0.3, axis="x")
    fig.suptitle(f"Band combinations of the {CLASS_LABELS[c]} multiplets - {opsim_name}")
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    save_fig(fig, f'band_combos_{c.replace("+", "plus")}')
    plt.show()

### Figure 5 - Band combinations compared between survey modes (`<run>_band_combos_heatmap.png`)

For the doublets, triplets and quadruplets: percentage of the multiplets **of the mode** that have a given band combination (each column sums to 100%, combinations below `min_pct` in every mode are hidden). A row that is bright in one column only is a combination specific to one mode; rows bright in several columns are shared.

In [ ]:
def combo_heatmap(ax, cls_value, min_pct=2.0):
    tab = combo_table(cls_value, "mode", MODES_USED)
    frac = 100 * tab.div(tab.sum(axis=0).replace(0, np.nan), axis=1)
    frac = frac.fillna(0.0)
    frac = frac[frac.max(axis=1) >= min_pct]
    order = tab.sum(axis=1).loc[frac.index].sort_values(ascending=False).index
    frac = frac.loc[order]
    if len(frac) == 0:
        ax.axis("off")
        return
    im = ax.imshow(
        frac.to_numpy(), cmap="magma_r", vmin=0, vmax=max(frac.to_numpy().max(), 1.0), aspect="auto"
    )
    for i in range(frac.shape[0]):
        for j in range(frac.shape[1]):
            val = frac.iloc[i, j]
            if val >= 1.0:
                ax.text(
                    j, i, f"{val:.0f}", ha="center", va="center", fontsize=10, color="w" if val > 50 else "k"
                )
    ax.set_xticks(range(frac.shape[1]))
    ax.set_xticklabels(list(frac.columns), rotation=30, ha="right")
    ax.set_yticks(range(frac.shape[0]))
    ax.set_yticklabels(list(frac.index))
    ax.set_title(CLASS_LABELS[cls_value])
    plt.colorbar(im, ax=ax, fraction=0.05, pad=0.02, label="% of the multiplets of the mode")


fig5, axes = plt.subplots(1, 3, figsize=(24, 8))
for ax, c in zip(axes, ["2", "3", "4"]):
    combo_heatmap(ax, c)
fig5.suptitle(f"Band combinations per survey mode - {opsim_name}")
fig5.tight_layout(rect=(0, 0, 1, 0.95))
save_fig(fig5, "band_combos_heatmap")
plt.show()

## 6. Time structure of the multiplets

### Figure 6 - Time between successive visits of a multiplet (`<run>_time_between_visits.png`)

For all the visits that have a predecessor in their multiplet, time elapsed since this predecessor, one curve per survey mode (fraction of the visits of the mode per logarithmic bin). The dashed line marks 33 minutes, the gap expected for the `pair_33` sequences; a second peak at a longer time would show the visits added later in the night (triplets). The short times (about one minute) are the DDF sequences.

In [ ]:
fig6, ax = plt.subplots(figsize=(13, 6.5))
bins = np.logspace(-0.5, 3, 140)
for m in MODES_USED:
    d = vis.loc[vis["mode"] == m, "dt_prev_min"].dropna().to_numpy()
    if len(d) < 100:
        continue
    ax.hist(
        d,
        bins=bins,
        weights=np.full(len(d), 1.0 / len(d)),
        histtype="step",
        lw=2.5,
        color=MODE_COLORS[m],
        label=f"{m} ({len(d):,})",
    )
ax.axvline(33, color="gray", ls="--", lw=1.5)
ax.text(33 * 1.05, ax.get_ylim()[1] * 0.6, "33 min", color="gray", fontsize=13)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Time since the previous visit of the same multiplet (min)")
ax.set_ylabel("Fraction of the visits of the mode per bin")
ax.grid(alpha=0.3, which="both")
ax.legend(title="mode (visits with a predecessor)")
ax.set_title(f"Time between successive visits of a multiplet - {opsim_name}")
fig6.tight_layout()
save_fig(fig6, "time_between_visits")
plt.show()

### Figure 7 - Number of multiplets per year and per class (`<run>_classes_per_year.png`)

One panel per survey mode, one stacked bar per year of the survey, colours = multiplicity classes. It shows whether the type of multiplet changes during the survey (for instance the first year, where the template tier takes pairs of visits, or the rolling cadence). The table is saved as `..._multiplets_per_year.csv`.

In [ ]:
mult["year_idx"] = np.clip(np.floor(mult["years"].to_numpy()).astype(int), 0, N_YEARS - 1)

fig7, axes = grid_axes(len(MODES_USED))
tables = {}
for ax, m in zip(axes, MODES_USED):
    sub = mult[mult["mode"] == m]
    tab = pd.crosstab(sub["year_idx"], sub["cls"]).reindex(
        index=range(N_YEARS), columns=CLASS_ORDER, fill_value=0
    )
    tables[m] = tab
    bottom = np.zeros(N_YEARS)
    for c in CLASS_ORDER:
        h = tab[c].to_numpy().astype(float)
        ax.bar(
            np.arange(N_YEARS) + 1,
            h,
            bottom=bottom,
            color=CLASS_COLORS[c],
            edgecolor="k",
            linewidth=0.3,
            label=CLASS_LABELS[c],
        )
        bottom = bottom + h
    ax.set_title(f"{m} ({len(sub):,})")
    ax.set_xlabel("Survey year")
    ax.set_ylabel("Number of multiplets")
    ax.set_xticks(np.arange(N_YEARS) + 1)
    ax.grid(alpha=0.3, axis="y")
handles, names = axes[0].get_legend_handles_labels()
fig7.legend(
    handles,
    names,
    ncol=len(CLASS_ORDER),
    loc="upper center",
    bbox_to_anchor=(0.5, 0.97),
    title="visits per multiplet",
)
fig7.suptitle(f"Multiplets per year - {opsim_name}")
fig7.tight_layout(rect=(0, 0, 1, 0.9))
save_fig(fig7, "classes_per_year")
plt.show()
pd.concat(tables, names=["mode", "year_idx"]).to_csv(
    os.path.join(DATA_DIR, f"{opsim_name}_multiplets_per_year.csv")
)

## 7. Cross-check with the scheduler sequences

The multiplets above are built from the pointings only. The `scheduler_note` tells how the scheduler itself named each visit (`pair_33`, `blob_long`, `greedy`, `templates`, ...). 

- `ct_family`: number of **visits** per scheduler family and per survey mode (saved as `..._families_per_mode.csv`).
- `signatures`: for each multiplicity class and mode, the combinations of scheduler families found in the multiplets (for example a triplet made of a `pair_33` and a later `blob_long` visit is `blob_long+pair_33`). It tells which scheduler sequences are behind the triplets and quadruplets (saved as `..._family_signatures.csv`).

### Figure 8 - Scheduler families in each survey mode (`<run>_families_per_mode.png`)

Fraction of the visits of each mode that belong to each scheduler family.

In [ ]:
ct_family = pd.crosstab(visits["family"], visits["mode"]).reindex(columns=MODES_USED, fill_value=0)
ct_family = ct_family.loc[ct_family.sum(axis=1).sort_values(ascending=False).index]
ct_family.to_csv(os.path.join(DATA_DIR, f"{opsim_name}_families_per_mode.csv"))

signatures = mult.groupby(["cls", "mode", "families"]).size().rename("N multiplets").reset_index()
signatures = signatures.sort_values(["cls", "mode", "N multiplets"], ascending=[True, True, False])
signatures.to_csv(os.path.join(DATA_DIR, f"{opsim_name}_family_signatures.csv"), index=False)
print("Most frequent scheduler-family signatures of the doublets and triplets:")
print(
    signatures[signatures["cls"].isin(["2", "3"])]
    .sort_values("N multiplets", ascending=False)
    .head(15)
    .to_string(index=False)
)

fam_names = list(ct_family.index)
fam_colors = plt.get_cmap("tab20")(np.linspace(0, 1, 20))
fig8, ax = plt.subplots(figsize=(15, 0.9 * len(MODES_USED) + 3.5))
left = np.zeros(len(MODES_USED))
for k, fam in enumerate(fam_names):
    w = 100 * ct_family.loc[fam].to_numpy() / ct_family.sum(axis=0).to_numpy()
    ax.barh(
        np.arange(len(MODES_USED)),
        w,
        left=left,
        color=fam_colors[k % 20],
        edgecolor="k",
        linewidth=0.4,
        label=fam,
    )
    for yi in range(len(MODES_USED)):
        if w[yi] >= 6:
            ax.text(left[yi] + w[yi] / 2, yi, f"{w[yi]:.0f}%", ha="center", va="center", fontsize=11)
    left = left + w
ax.set_yticks(np.arange(len(MODES_USED)))
ax.set_yticklabels([f"{m} ({ct_family[m].sum():,})" for m in MODES_USED])
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Fraction of the visits of the mode (%)")
ax.legend(title="scheduler family", loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=11)
ax.set_title(f"Scheduler families per survey mode - {opsim_name}")
ax.grid(alpha=0.3, axis="x")
fig8.tight_layout()
save_fig(fig8, "families_per_mode")
plt.show()
ct_family

## 8. Where are the multiplets? HEALPix maps

A visit covers all the pixels within `FOV_RADIUS_DEG` of its pointing centre (a KD-tree on the pixel centres, as the MAF HealpixSlicer does without the camera footprint). The map of a class counts, for each pixel, **the number of multiplets of this class covering the pixel** over the 10 years (the multiplet is located at the position of its first visit). One pass over the multiplets fills the maps of all classes, and the map of the total number of visits in the multiplets (`wmap`), used for the mean multiplicity.

`MAP_MODE` (configuration) restricts the maps to one survey mode, e.g. `'WFD'` or `'DDF'`; `None` uses all the multiplets.

In [ ]:
tree_pix = cKDTree(np.array(hp.pix2vec(NSIDE, np.arange(NPIX))).T)
chord_fov = 2.0 * np.sin(np.radians(FOV_RADIUS_DEG) / 2.0)


def coverage_maps(ra_deg, dec_deg, class_code, weights, n_class, chunk=200000):
    # cov[k, pix]: number of multiplets of class k covering the pixel; wmap[pix]: sum of the weights of the multiplets covering it
    cov = np.zeros((n_class, NPIX))
    wmap = np.zeros(NPIX)
    for s in range(0, len(ra_deg), chunk):
        sl = slice(s, s + chunk)
        v = hp.ang2vec(ra_deg[sl], dec_deg[sl], lonlat=True)
        lists = tree_pix.query_ball_point(v, chord_fov, workers=-1)
        lens = np.fromiter((len(x) for x in lists), dtype=np.int64, count=len(lists))
        idx = np.fromiter(itertools.chain.from_iterable(lists), dtype=np.int64, count=int(lens.sum()))
        cc = np.repeat(class_code[sl], lens)
        ww = np.repeat(weights[sl], lens)
        cov += np.bincount(cc * NPIX + idx, minlength=n_class * NPIX).reshape(n_class, NPIX)
        wmap += np.bincount(idx, weights=ww, minlength=NPIX)
    return cov, wmap


sel = np.ones(len(mult), dtype=bool) if MAP_MODE is None else (mult["mode"] == MAP_MODE).to_numpy()
class_index = {c: i for i, c in enumerate(CLASS_ORDER)}
class_code = mult["cls"].map(class_index).to_numpy()
cov, wmap = coverage_maps(
    mult["ra"].to_numpy()[sel],
    mult["dec"].to_numpy()[sel],
    class_code[sel],
    mult["n"].to_numpy()[sel].astype(float),
    len(CLASS_ORDER),
)
cov_total = cov.sum(axis=0)
map_tag = "all" if MAP_MODE is None else MAP_MODE.replace(" ", "_").replace("(", "").replace(")", "")
print(
    f"Maps for {sel.sum()} multiplets (mode: {MAP_MODE}); {int((cov_total > 0).sum())} pixels covered of {NPIX}"
)
np.save(os.path.join(DATA_DIR, f"{opsim_name}_multiplet_maps_{map_tag}.npy"), cov)

### Figure 9 - Number of multiplets per pixel and per class (`<run>_maps_counts_<mode>.png`)

One map per multiplicity class and one for all the multiplets: number of multiplets covering each pixel over the 10 years (grey = no multiplet). The maps are saved in `DATA13_Multiplets/` as an array `(5 classes, npix)` (`..._multiplet_maps_<mode>.npy`).

In [ ]:
def mollview_panel(fig, sub, m, valid, title, unit, vmin=None, vmax=None, cmap="viridis"):
    mm = np.array(m, dtype=float)
    mm[~valid] = hp.UNSEEN
    hp.mollview(mm, fig=fig.number, sub=sub, title=title, unit=unit, min=vmin, max=vmax, cmap=cmap)
    hp.graticule(dpar=30, dmer=60)


fig9 = plt.figure(figsize=(21, 10))
panels = [(CLASS_LABELS[c], cov[k]) for k, c in enumerate(CLASS_ORDER)] + [("all multiplets", cov_total)]
for k, (title, m) in enumerate(panels):
    valid = m > 0
    vmax = np.percentile(m[valid], 99) if valid.any() else None
    mollview_panel(fig9, (2, 3, k + 1), m, valid, title, "multiplets / pixel", vmin=0, vmax=vmax)
fig9.suptitle(f"Number of multiplets covering each pixel (mode: {MAP_MODE}) - {opsim_name}", fontsize=18)
save_fig(fig9, f"maps_counts_{map_tag}")
plt.show()

### Figure 10 - Fraction of each class and mean multiplicity per pixel (`<run>_maps_fractions_<mode>.png`)

For the pixels covered by at least `MIN_TOTAL_PIX` multiplets: fraction (%) of the multiplets covering the pixel that are singlets, doublets, triplets, and quadruplets or more (classes `4` and `5+` merged), and the **mean number of visits per multiplet**. These maps show directly which kind of multiplet dominates each region of the sky, hence the link between the multiplet type and the footprint (WFD, mini-surveys, DDF fields, ...). Compare with the footprint map of `08_Footprint_TaskForce`.

In [ ]:
valid_f = cov_total >= MIN_TOTAL_PIX
safe_total = np.where(cov_total > 0, cov_total, 1.0)
frac_maps = [
    (CLASS_LABELS["1"], 100 * cov[0] / safe_total),
    (CLASS_LABELS["2"], 100 * cov[1] / safe_total),
    (CLASS_LABELS["3"], 100 * cov[2] / safe_total),
    ("4 visits or more", 100 * (cov[3] + cov[4]) / safe_total),
]

fig10 = plt.figure(figsize=(21, 10))
for k, (title, m) in enumerate(frac_maps):
    mollview_panel(
        fig10,
        (2, 3, k + 1),
        m,
        valid_f,
        f"Fraction of {title}",
        "% of the multiplets",
        vmin=0,
        vmax=100,
        cmap="magma_r",
    )
mean_n = wmap / safe_total
mollview_panel(
    fig10,
    (2, 3, 5),
    mean_n,
    valid_f,
    "Mean number of visits per multiplet",
    "visits",
    vmin=1,
    vmax=max(np.percentile(mean_n[valid_f], 99), 2.0) if valid_f.any() else 2.0,
    cmap="viridis",
)
fig10.suptitle(
    f"Class fractions and mean multiplicity per pixel (mode: {MAP_MODE}) - {opsim_name}", fontsize=18
)
save_fig(fig10, f"maps_fractions_{map_tag}")
plt.show()

## 9. Reading the results

- **Type of multiplet given the mode**: Figures 1 and 2 (stacked bars) and the `% of the mode` columns of the summary table.
- **Mode given the type**: the `% of the class` columns; a class living in one mode only is a signature of this mode.
- **Strength of the association**: Cramer V and the standardised residuals of Figure 3. Residuals larger than a few tens of sigma are expected with this statistics; look at the **sign and the pattern**, and at V.
- **Band combinations**: Figures 4 and 5. Compare the combinations between modes (for example the same-filter pairs `uu`, `gg`, `rr`, `yy` versus the pairs in two different filters).
- **Time structure**: Figure 6 (gap between visits) and Figure 7 (evolution with the year).
- **Where**: Figures 9 and 10, to be compared with the footprint map of `08_Footprint_TaskForce`.

**Points of attention**

1. The definition of a multiplet (same night, same pointing) merges, in one multiplet, visits that the scheduler may have requested in separate sequences (for example a pair and a later visit of the same field). The signatures of section 7 show when it happens.
2. A multiplet is located at the position of its first visit; the pixels it covers are all the pixels within `FOV_RADIUS_DEG`, so the maps are smooth at the scale of a field of view (the dithering between nights widens the coverage in the WFD).
3. The DDF multiplets are by construction long (many visits of one DDF in one night): the class `5+` is expected to be dominated by them.
4. Multiplets of one night are not independent from one night to the next (the scheduler revisits the same fields), so the p-values of the chi-square test are optimistic: use the effect size.

**Possible extensions**: comparison with another run (change `opsdb`); one multiplet type per filter pair as a function of the Moon phase; time since the previous multiplet of the same field (revisit time) per class and mode.